# Remote VS Code on Kaggle (code-server + ngrok)

A small, self-contained replacement for the old, unmaintained **ColabCode**.
It runs [`code-server`](https://github.com/coder/code-server) (VS Code in the browser)
on a Kaggle kernel and exposes it through an **ngrok** tunnel using ngrok's *current* API.

## One-time setup
1. **Enable Internet** for this notebook: right sidebar -> **Settings** -> toggle **Internet** on.
2. **Add your ngrok token as a Secret** (do NOT paste it in a code cell):
   - Get a token: <https://dashboard.ngrok.com/get-started/your-authtoken>
   - Right sidebar -> **Add-ons** -> **Secrets** -> add a secret named `NGROK_TOKEN`.
   - Make sure the secret is **attached** to this notebook (toggle it on).
3. Run the two code cells below in order.

## Notes
- The `extensions.coder.com` warnings you may have seen elsewhere are gone here
  (we don't pre-install Coder's dead extension marketplace).
- ngrok's free tier allows **one** active tunnel; the launcher kills stale sessions first.
- Kaggle kernels are temporary: when the session stops, the tunnel and editor stop too.

## 1. Install dependencies

In [ ]:
# Install code-server (VS Code in the browser) and the ngrok Python client
!curl -fsSL https://code-server.dev/install.sh | sh
!pip install -q pyngrok

## 2. Launch code-server and open the tunnel

In [ ]:
import os
import subprocess
import time

from pyngrok import ngrok

# ---- Config -------------------------------------------------------------
PORT = 10000
PASSWORD = "change-me"  # <-- set your own code-server login password

# ---- Get the ngrok token from Kaggle Secrets (never hardcode it) --------
NGROK_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient

    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")

if not NGROK_TOKEN:
    raise RuntimeError(
        "No ngrok token found. In Kaggle: Add-ons -> Secrets, add a secret named "
        "'NGROK_TOKEN' with your token from "
        "https://dashboard.ngrok.com/get-started/your-authtoken and attach it to this notebook."
    )

# ---- Open the tunnel (modern pyngrok / ngrok API) -----------------------
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()  # clear any stale session (free tier allows only one tunnel)
public_url = ngrok.connect(PORT, "http").public_url

print("=" * 64)
print(" Code Server URL :", public_url)
print(" Password        :", PASSWORD)
print("=" * 64)

# ---- Launch code-server in the background -------------------------------
proc = subprocess.Popen(
    ["code-server", "--bind-addr", f"0.0.0.0:{PORT}", "--auth", "password"],
    env={**os.environ, "PASSWORD": PASSWORD},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(5)
print(f"code-server started (pid {proc.pid}). Open the URL above and log in with the password.")

## 3. (Optional) Keep the kernel alive

Kaggle stops idle kernels. Run this cell to keep the session (and your editor) alive
while you work. Interrupt the cell when you're done.

In [ ]:
import time

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopped keep-alive. code-server is still running until the kernel stops.")